
## 1. 模型调用的分类
角度1:模型功能
- 对话模型(LLMS,text,Model) (推荐)
- 非对话模型(Chat Models)
- 嵌入模型(Embedding Models) (RAG相关)

角度2:安装模型调用
- 硬编码
- 环境变量
- 配置文件(推荐)

角度3:具体API调用
- openAI提供的
- 其他大模型提供的
- LangChain统一方式调用API(推荐)

### 2.非对话模型
- 不支持多轮对话模型
- 一输入一输出

### 对话模型
- 支持多轮对话


In [2]:
print("pink")

pink


In [1]:
# 阻塞式非流式调用输出
import os 
import dotenv
dotenv.load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_BASE"] = os.getenv("OPENAI_BASE_URL")

chat_model = ChatOpenAI(
    model= os.getenv("MODEL_NAME", "gpt-3.5-turbo"),
)

messages = [
    # SystemMessage(content="你是一个助手，请回答我的问题。"),
    HumanMessage(content="你好，你是谁？"),
]

# 非流式调用
response = chat_model.invoke(messages)

print(response ,type(response))

content='Sorry, to prevent abuse of free resources, accounts that have not been recharged can only try 10 times. You can increase the free quota after recharging; https://console.aihubmix.com/topup' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 0, 'prompt_tokens': 0, 'total_tokens': 0, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gpt-5.5-free', 'system_fingerprint': None, 'id': 'chatcmpl-fake-1783740931330851219', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f4f3e-9ba6-7c92-b03a-cb0e20a2cb03-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0, 'input_token_details': {}, 'output_token_details': {}} <class 'langchain_core.messages.ai.AIMessage'>


接下来看看流式输出（阻塞）
流式输出是指，模型会返回一个流，然后客户端会不断读取这个流，直到流结束。

In [2]:
import os 
import dotenv
dotenv.load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_BASE"] = os.getenv("OPENAI_BASE_URL")

chat_model = ChatOpenAI(
    model= os.getenv("MODEL_NAME", "gpt-3.5-turbo"),
    streaming=True,
)

messages = [
    # SystemMessage(content="你是一个助手，请回答我的问题。"),
    HumanMessage(content="你好，你是谁？"),
]

# # 非流式调用
# response = chat_model.invoke(messages)

print("开始流式输出：")

for chunk in chat_model.stream(messages): 
    print(chunk.content, end="", flush=True) # 输出流式数据,刷新缓冲区，无换行符


print("\n流式输出结束")

开始流式输出：


PermissionDeniedError: <!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><meta http-equiv="content-security-policy" content="default-src &#39;none&#39;; script-src &#39;nonce-OCj6ft9Bm1F7rusCMYy9oq&#39; &#39;unsafe-eval&#39; https://challenges.cloudflare.com; script-src-attr &#39;none&#39;; style-src &#39;unsafe-inline&#39;; img-src &#39;self&#39; https://challenges.cloudflare.com; connect-src &#39;self&#39; https://challenges.cloudflare.com; frame-src &#39;self&#39; https://challenges.cloudflare.com blob:; child-src &#39;self&#39; https://challenges.cloudflare.com blob:; worker-src blob:; form-action http: https:; base-uri &#39;self&#39;"><style>*{box-sizing:border-box;margin:0;padding:0}html{line-height:1.15;-webkit-text-size-adjust:

批量调用，

In [1]:
import os 
import dotenv
dotenv.load_dotenv()
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_BASE"] = os.getenv("OPENAI_BASE_URL")

chat_model = ChatOpenAI(
    model= os.getenv("MODEL_NAME", "gpt-3.5-turbo"),
    streaming=True,
)

messages = [
    # SystemMessage(content="你是一个助手，请回答我的问题。"),
    HumanMessage(content="你好，你是谁？"),
]

messages1 = [
    # SystemMessage(content="你是一个助手，请回答我的问题。"),
    HumanMessage(content="背诵静夜思"),
]

# # 非流式调用
# response = chat_model.invoke(messages)

# print("开始流式输出：")

# for chunk in chat_model.stream(messages): 
#     print(chunk.content, end="", flush=True) # 输出流式数据,刷新缓冲区，无换行符


# print("\n流式输出结束")


# 批量调用
response = chat_model.batch([messages, messages1])
print(response)

[AIMessage(content='Sorry, to prevent abuse of free resources, accounts that have not been recharged can only try 10 times. You can increase the free quota after recharging; https://console.aihubmix.com/topup', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-5.5-free', 'model_provider': 'openai'}, id='lc_run--019f4b85-56ec-7f70-a49d-dfacb34f5416', tool_calls=[], invalid_tool_calls=[]), AIMessage(content='Sorry, to prevent abuse of free resources, accounts that have not been recharged can only try 10 times. You can increase the free quota after recharging; https://console.aihubmix.com/topup', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-5.5-free', 'model_provider': 'openai'}, id='lc_run--019f4b85-56ed-71d2-a004-9cc18290989f', tool_calls=[], invalid_tool_calls=[])]


## 提示词模板 prompt template
变量 + prompt
